# Customer Churn Analytics
Objective: calculate baseline churn, revenue-based CLV proxy, MRR at risk, segment churn rates, and behavioral signals.

**Input:** `customer_churn_raw_v2 (1).csv` (upload it to the notebook session, or update `INPUT_FILE`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

INPUT_FILE = "customer_churn_raw_v2 (1).csv"
raw = pd.read_csv(INPUT_FILE)
print("Raw shape:", raw.shape)
display(raw.head())
print("\\nMissing values:")
display(raw.isna().sum().to_frame("missing_count"))
print("Exact duplicate rows:", raw.duplicated().sum())

## 1. Prepare data
Standardize column headers and remove exact duplicate rows so repeated records do not inflate the metrics. `TotalChargesINR` contains text placeholders, so convert it to numeric for quality inspection.

In [ ]:
df = raw.copy()
df.columns = (df.columns.astype(str).str.strip()
    .str.replace(r"([a-z0-9])([A-Z])", r"\\1_\\2", regex=True)
    .str.replace(r"[^A-Za-z0-9]+", "_", regex=True)
    .str.strip("_").str.lower())
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
df["total_charges_inr"] = pd.to_numeric(df["total_charges_inr"], errors="coerce")
df["churn_flag"] = df["churn"].astype(str).str.strip().str.lower().eq("yes").astype(int)
df["tenure_band"] = pd.cut(df["tenure_months"], [-1,6,12,24,36,48,10**6],
    labels=["0–6 months","7–12 months","13–24 months","25–36 months","37–48 months","49+ months"])
print("Exact duplicates removed:", duplicate_count)
print("Analysis rows:", len(df))
print("Missing region values:", df["region"].isna().sum())
print("Non-numeric/missing total charges:", df["total_charges_inr"].isna().sum())

## 2. Baseline churn, CLV proxy, and MRR at risk

**CLV proxy formula:** average monthly charges × average observed tenure. It is revenue-only, not profit CLV, because margin and acquisition-cost data are unavailable. **MRR at risk:** sum of monthly charges for customers marked as churned.

In [ ]:
n = len(df)
churned = int(df["churn_flag"].sum())
churn_rate = churned / n
avg_monthly = df["monthly_charges_inr"].mean()
avg_tenure = df["tenure_months"].mean()
clv_proxy = avg_monthly * avg_tenure
total_mrr = df["monthly_charges_inr"].sum()
mrr_loss = df.loc[df["churn_flag"].eq(1), "monthly_charges_inr"].sum()

print(f"Customers: {n}")
print(f"Churned customers: {churned}")
print(f"Baseline churn rate: {churn_rate:.1%}")
print(f"Average monthly charges: ₹{avg_monthly:,.2f}")
print(f"Average observed tenure: {avg_tenure:.2f} months")
print(f"Revenue-based CLV proxy: ₹{clv_proxy:,.2f}")
print(f"Total monthly charges: ₹{total_mrr:,.2f}")
print(f"MRR at risk from churned records: ₹{mrr_loss:,.2f} ({mrr_loss/total_mrr:.1%})")

## 3. Segment churn rates
Compare churn by contract type, tenure band, and payment method.

In [ ]:
def churn_summary(column):
    result = df.groupby(column, dropna=False, observed=False).agg(
        customers=("churn_flag", "size"),
        churned=("churn_flag", "sum"),
        churn_rate=("churn_flag", "mean"),
        monthly_charges=("monthly_charges_inr", "sum")
    ).reset_index()
    result["churn_rate_pct"] = result["churn_rate"] * 100
    return result

contract_summary = churn_summary("contract_type")
tenure_summary = churn_summary("tenure_band")
payment_summary = churn_summary("payment_method")
display(contract_summary)
display(tenure_summary)
display(payment_summary)

In [ ]:
for column, title in [
    ("contract_type", "Churn rate by contract type"),
    ("tenure_band", "Churn rate by tenure"),
    ("payment_method", "Churn rate by payment method")
]:
    temp = churn_summary(column)
    plt.figure(figsize=(8,4))
    plt.bar(temp[column].astype(str), temp["churn_rate_pct"])
    plt.title(title)
    plt.ylabel("Churn rate (%)")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## 4. Boxplot and correlation heatmap
Correlations are descriptive associations, not proof of causation.

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="churn", y="tenure_months")
plt.title("Tenure distribution by churn status")
plt.tight_layout()
plt.show()

numeric = ["age","tenure_months","monthly_charges_inr","total_charges_inr",
           "support_tickets","late_payments","churn_flag"]
corr = df[numeric].corr(numeric_only=True)
plt.figure(figsize=(9,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap")
plt.tight_layout()
plt.show()
print("Correlation with churn:")
display(corr["churn_flag"].sort_values(ascending=False).to_frame("correlation_with_churn"))

## 5. Business recommendations
1. Prioritize onboarding and proactive check-ins during the first six months.
2. Test retention offers or flexible benefits for month-to-month customers.
3. Trigger support follow-up when customers submit repeated tickets.
4. Offer clear payment reminders and payment-failure recovery for late payments.
5. Investigate Wallet and UPI experiences; test improvements rather than assuming payment method causes churn.
6. Measure impact with a holdout group and track churn, MRR retained, and intervention cost.